# Model Training & Comparison — Customer Churn
Interactive companion to `src/modeling/train_models.py`. Trains 4 classifiers and compares them on the full metric suite.

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

df = pd.read_csv('../data/processed/telecom_customers_features.csv')
df.shape

(12000, 35)

## Prepare features

In [2]:
feature_cols = ['tenure','MonthlyCharges','TotalCharges','SupportTicketsLastYear',
    'ContractDurationMonths','TotalAddOnServices','CustomerValueScore','IsAutoPay',
    'HasInternetNoProtection','IsMultiServiceHousehold','SeniorCitizen','gender',
    'Partner','Dependents','InternetService','Contract','PaymentMethod','PaperlessBilling']
X = df[feature_cols].copy(); y = df['ChurnFlag']
cat_cols = X.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    X[c] = LabelEncoder().fit_transform(X[c].astype(str))
num_cols = [c for c in feature_cols if c not in cat_cols]
X[num_cols] = StandardScaler().fit_transform(X[num_cols])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape

/tmp/ipykernel_631/2232954640.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns.tolist()


((9600, 18), (2400, 18))

## Train & compare 4 models

In [3]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
        scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(), eval_metric='logloss', random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]
    results.append({'model': name, 'accuracy': accuracy_score(y_test,pred),
        'precision': precision_score(y_test,pred), 'recall': recall_score(y_test,pred),
        'f1': f1_score(y_test,pred), 'roc_auc': roc_auc_score(y_test,proba)})

results_df = pd.DataFrame(results).sort_values('roc_auc', ascending=False).round(4)
results_df

,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.7067,0.7826,0.7124,0.7458,0.7884
2,Random Forest,0.7092,0.7690,0.7414,0.7549,0.7819
3,XGBoost,0.6917,0.7722,0.6945,0.7313,0.7755
1,Decision Tree,0.6954,0.7837,0.6848,0.7310,0.7739


## Model selection rationale
The model with the highest ROC-AUC is selected, tie-broken by recall — in this business, a **missed churner (false negative) costs far more** than an unnecessary retention offer (false positive), so recall is weighted heavily.

Full training pipeline (with persisted artifacts, confusion matrices, ROC curves, and feature importance plots) lives in `src/modeling/train_models.py` and produces `reports/Model_Comparison_Report.md`.